# Research-Informed Sequential Models for Gender-Based Violence Tweet Classification

**Formative Assignment 2 — Sequential Models for NLP**

**Challenge:** Gender-Based Violence Tweet Classification Challenge (Zindi)

**Research question:** *How effectively can sequential modelling approaches be used to classify tweets about gender-based violence into violence types, and what evidence supports the strengths and limitations of the selected approaches?*

### Contents
0. Setup
1. Problem and Data Investigation
2. Related Work and Sequential Model Selection
3. Experimental Investigation
4. Error Analysis and Limitations

## 0. Setup

The notebook is designed to run on Google Colab (GPU runtime recommended for Section 3). Upload `Train.csv`, `Test.csv` and `SampleSubmission.csv` to the session, or place them in a Google Drive folder and set `DRIVE_DIR` below.

In [ ]:
import os, re, random, collections, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
DRIVE_DIR = "/content/drive/MyDrive/gbv_formative"  # change if your data lives elsewhere

if IN_COLAB and not os.path.exists("Train.csv"):
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_DIR
else:
    DATA_DIR = "."

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{name}.png"), bbox_inches="tight")

print("Colab:", IN_COLAB, "| data dir:", DATA_DIR)

## 1. Problem and Data Investigation

### 1.1 The problem and its practical relevance

Gender-based violence (GBV) is widespread across African countries and globally, yet much of it goes unreported through formal channels because of stigma, fear of retaliation and weak reporting infrastructure. Survivors, witnesses and news outlets increasingly discuss these experiences on social media. Automatically categorising such posts by **type of violence** — *sexual, physical, emotional, economic,* or *harmful traditional practice* (e.g. FGM, child marriage) — can help NGOs, helplines and public-health researchers monitor trends, route content to the right support services, and understand forms of violence (such as economic or emotional abuse) that are under-represented in official statistics.

The challenge explicitly asks for a classifier that works *"without using keywords"*. This matters: a useful system must understand **how words are used in context** (who did what to whom, literal vs. figurative use, reported news vs. personal disclosure), which is precisely what sequential models are designed to capture. Tweets are sequences of tokens, and the meaning of a GBV cue often depends on its neighbours — *"he beats me"* vs. *"that beats me"*, *"I got fired because I refused to sleep with him"* (economic) vs. a sexual-assault disclosure.

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "Train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "Test.csv"))

print("Train:", train.shape, "| Test:", test.shape)
print("Missing values:", train.isna().sum().to_dict())
print("Duplicate tweets in train:", train.tweet.duplicated().sum())
print("Duplicate tweets with conflicting labels:", (train.groupby("tweet").type.nunique() > 1).sum())
print("Train/test tweet overlap:", test.tweet.isin(set(train.tweet)).sum())
train.head()

### 1.2 Class distribution

In [ ]:
LABELS = train.type.value_counts().index.tolist()
counts = train.type.value_counts()
dist = pd.DataFrame({"count": counts, "percent": (counts / len(train) * 100).round(2)})
display(dist)
print(f"Imbalance ratio (largest / smallest): {counts.max() / counts.min():.0f} : 1")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(x=counts.values, y=counts.index, ax=axes[0], palette="viridis")
axes[0].set_title("Class counts (linear scale)")
sns.barplot(x=counts.values, y=counts.index, ax=axes[1], palette="viridis")
axes[1].set_xscale("log")
axes[1].set_title("Class counts (log scale)")
for ax in axes:
    ax.set_xlabel("Number of tweets"); ax.set_ylabel("")
savefig("class_distribution")
plt.show()

**What this reveals.** The data is extremely imbalanced: sexual violence accounts for ~82% of tweets, physical violence ~15%, while emotional (1.6%), economic (0.55%) and harmful traditional practice (0.47%) together make up under 3%. The largest-to-smallest ratio is ~174:1.

**Implications for modelling.**
- **Accuracy is misleading.** Always predicting *sexual_violence* already yields ~82% accuracy; the starter Naive Bayes notebook reaches 88% while never predicting two of the five classes. We will therefore use **macro-F1** as the primary metric (every class weighted equally), and report accuracy (the Zindi leaderboard metric), per-class F1 and confusion matrices alongside it.
- **Imbalance handling is required** for all models: class-weighted loss functions (and/or re-sampling) so minority classes influence training.
- **Stratified splits** are needed so that the ~40–70 minority examples per validation fold are representative; with so few examples, minority-class scores will be noisy and we should interpret small differences cautiously.

### 1.3 Sequence length

Sequence length determines the input length (`max_len`) for neural sequence models and how far dependencies may need to be carried. We look at word counts and at sub-word token counts (what a Transformer actually sees).

In [ ]:
train["n_words"] = train.tweet.str.split().str.len()
test["n_words"] = test.tweet.str.split().str.len()
train["n_chars"] = train.tweet.str.len()

display(train.groupby("type")[["n_words", "n_chars"]].describe(percentiles=[.5, .95]).round(1))
print("Word-count percentiles (train):", train.n_words.quantile([.5, .9, .95, .99]).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=train, x="n_words", y="type", order=LABELS, ax=axes[0], palette="viridis")
axes[0].set_title("Words per tweet by class"); axes[0].set_ylabel("")
sns.kdeplot(train.n_words, label="train", ax=axes[1], fill=True)
sns.kdeplot(test.n_words, label="test", ax=axes[1], fill=True)
axes[1].set_title("Words per tweet: train vs test"); axes[1].legend()
savefig("sequence_length")
plt.show()

In [ ]:
# Sub-word token lengths as seen by a Transformer tokenizer
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    train["n_subwords"] = [len(x) for x in tok(train.tweet.tolist(), add_special_tokens=True)["input_ids"]]
    print("Sub-word token percentiles:", train.n_subwords.quantile([.5, .95, .99, 1.0]).to_dict())
    print("Share of tweets longer than 128 sub-words:", (train.n_subwords > 128).mean().round(4))
except Exception as e:
    print("Tokenizer unavailable:", e)

**What this reveals.** Tweets are short (median 43 words, 99th percentile 61 words, max 68; median 56 and 99th percentile 83 sub-word tokens, only 1 tweet above 128) — consistent with Twitter's character limit. Physical-violence tweets are notably shorter (median ~19 words, mean 23 vs. 42 for sexual violence) because many are **news headlines** (e.g. *"My Husband Beats Me Frequently, Wife Tells Court"*), whereas sexual, emotional and economic tweets tend to be longer, first-person narratives. The train and test length distributions are similar (test slightly shorter).

**Implications for modelling.**
- A `max_len` of ~64 words (or 128 sub-word tokens) covers essentially every tweet without truncation, so padding cost is low and the full context is always available.
- Sequences of 20–60 tokens are within the range where gated RNNs (LSTM/GRU) can retain dependencies, while self-attention can relate any two tokens directly. Length alone is a weak but real signal (headline style ↔ physical violence), which models may exploit.

### 1.4 Vocabulary and noise characteristics

In [ ]:
def tokenize(s):
    return re.findall(r"[a-z']+", s.lower())

vocab = collections.Counter(w for s in train.tweet for w in tokenize(s))
hapax = sum(1 for c in vocab.values() if c == 1)
print(f"Vocabulary size: {len(vocab):,} | words seen once: {hapax:,} ({hapax/len(vocab):.1%})")

test_vocab = collections.Counter(w for s in test.tweet for w in tokenize(s))
oov_tokens = sum(c for w, c in test_vocab.items() if w not in vocab)
print(f"Test tokens out-of-vocabulary w.r.t. train: {oov_tokens / sum(test_vocab.values()):.2%}")

EMOJI = r"[\U0001F300-\U0001FAFF\u2600-\u27BF]"
noise = {
    "URL": r"http\S+|www\.\S+",
    "@mention": r"@\w+",
    "#hashtag": r"#\w+",
    "HTML entity (&amp; etc.)": r"&\w+;",
    "emoji": EMOJI,
    "non-ASCII char": r"[^\x00-\x7F]",
    "elongated word (sooo)": r"(\w)\1{2,}",
    "ALL-CAPS word (4+ letters)": r"\b[A-Z]{4,}\b",
    "censored word (s**, f*ck)": r"\w\*+\w*",
}
noise_df = pd.DataFrame({k: train.tweet.str.contains(p, regex=True) for k, p in noise.items()})
display((noise_df.mean() * 100).round(2).rename("% of tweets").to_frame())

# Zipf plot
freqs = np.array(sorted(vocab.values(), reverse=True))
plt.figure(figsize=(6, 4))
plt.loglog(np.arange(1, len(freqs) + 1), freqs)
plt.xlabel("Rank"); plt.ylabel("Frequency"); plt.title("Word frequency vs rank (Zipf)")
savefig("zipf")
plt.show()

**What this reveals.** The vocabulary is ~37k word types, about half of which occur only once — a long Zipfian tail of rare words, names, misspellings and non-English tokens. URLs, @mentions and hashtags have already been removed by the organisers, but other social-media noise remains: HTML entities (`&amp;`) in ~8% of tweets, emojis, elongations, ALL-CAPS emphasis and self-censored words (*"rough s** with him"*).

**Implications for modelling.**
- Light normalisation is useful (unescape HTML entities, lower-casing for word-level models), but we should **not** aggressively strip emojis, capitalisation or censored words, since they carry emotional tone.
- The long tail means word-level models will see many unknown/rare tokens. **Pre-trained embeddings** (GloVe/fastText) and especially **sub-word tokenisation** (WordPiece/BPE in Transformers) mitigate this, which supports including a pre-trained Transformer.

### 1.5 Class-specific vocabulary: how was the data collected?

We look at the most characteristic words and bigrams of each class, using a log-odds ratio of each term's frequency inside vs. outside the class.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(ngram_range=(1, 2), min_df=5, token_pattern=r"[a-z']{2,}")
X_counts = cv.fit_transform(train.tweet.str.lower())
terms = np.array(cv.get_feature_names_out())

rows = []
for lab in LABELS:
    m = (train.type == lab).values
    in_c = np.asarray(X_counts[m].sum(0)).ravel() + 1
    out_c = np.asarray(X_counts[~m].sum(0)).ravel() + 1
    log_odds = np.log(in_c / in_c.sum()) - np.log(out_c / out_c.sum())
    top = np.argsort(-log_odds)[:12]
    rows.append({"class": lab, "most characteristic terms": ", ".join(terms[top])})
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(rows))

In [ ]:
# How far can five hand-written keyword rules go?
from sklearn.metrics import accuracy_score, f1_score, classification_report

KEYWORD_RULES = [  # applied in order, first match wins
    ("Harmful_Traditional_practice", r"genital|fgm|child marriage|forced marriage|forced into marriage|marry"),
    ("economic_violence", r"fired|job"),
    ("emotional_violence", r"humiliat|insult"),
    ("Physical_violence", r"beat"),
    ("sexual_violence", r"rap|sexual"),
]

def keyword_rule(texts):
    s = pd.Series(texts).str.lower()
    pred = pd.Series("sexual_violence", index=s.index)
    done = pd.Series(False, index=s.index)
    for lab, pat in KEYWORD_RULES:
        m = s.str.contains(pat) & ~done
        pred[m] = lab
        done |= m
    return pred.values, done.values

rule_pred, covered = keyword_rule(train.tweet)
print(f"Tweets matched by at least one rule: {covered.mean():.1%}")
print(f"Accuracy: {accuracy_score(train.type, rule_pred):.4f} | Macro-F1: {f1_score(train.type, rule_pred, average='macro'):.4f}")
print(classification_report(train.type, rule_pred, digits=3))

**What this reveals — a keyword-collection artefact.** Each class is dominated by one or two query phrases: *"raped"* (sexual), *"husband beats"* (physical), *"humiliated/insulted … in public"* (emotional), *"fired because … job"* (economic), and *"female genital mutilation / child / forced marriage"* (harmful traditional practice). This strongly suggests the tweets were **collected by keyword search**, and the label was largely determined by which query retrieved them.

As a result, **five regular expressions achieve ~95% accuracy on the training set** — higher than the starter Naive Bayes model — without any learning. However, their **macro-F1 is only ~0.77**: precision collapses for economic violence (0.31) and harmful traditional practice (0.52), because those keywords also appear in tweets labelled with another class.

**Implications for modelling.**
- High accuracy is easy and is *not* evidence that a model understands GBV. The meaningful difficulty lies in the **~5% of tweets where keywords are absent, figurative, or point to more than one class** — this is where contextual, sequential modelling should add value, and it is where our error analysis (Section 4) will focus.
- Bag-of-words baselines will be very strong on this data; we expect small gaps between models on accuracy, and should compare them on macro-F1 and minority-class F1.
- Models may learn shortcuts that would not generalise to tweets collected differently. We will note this as a key limitation and probe it (e.g. by masking the query keywords) in Section 4.

### 1.6 Ambiguity, overlapping cues and label noise

In [ ]:
lower = train.tweet.str.lower()
cue_hits = pd.DataFrame({lab: lower.str.contains(pat) for lab, pat in KEYWORD_RULES})
train["n_cue_classes"] = cue_hits.sum(1)
print("Number of different class cues present in a tweet:")
display(train.n_cue_classes.value_counts().sort_index().rename("tweets").to_frame())

multi = train[train.n_cue_classes >= 2]
print("Label distribution of tweets containing cues of 2+ classes:")
display(multi.type.value_counts().to_frame())

print("\nExamples with cues from several classes:")
for _, r in multi.sample(6, random_state=SEED).iterrows():
    print(f"[{r.type}] {r.tweet[:220]}")

print("\nPossible label noise — figurative 'beats' labelled physical violence:")
fig_beat = train[(train.type == "Physical_violence") & lower.str.contains(r"beats me to it|it beats|beats the|beats all")]
print(len(fig_beat), "tweets")
for t in fig_beat.tweet.head(4):
    print(" -", t[:200])

**What this reveals.** About 2,150 tweets (5.4%) contain cues from two or more classes. Most describe co-occurring abuse (*"raped and beat me"*) and are labelled *sexual_violence*, suggesting annotators prioritised sexual violence when several types were present; others, such as *"my boss fired me because I refused to have sex with him"*, are labelled *economic*. Deciding the label therefore requires understanding the **relationship between the events** in the sentence (what is the main harm, what caused what), not the presence of a single word. We also find **label noise**: at least 313 tweets that use *"beats"* figuratively (*"it beats me why…"*, *"nothing beats…"*) are labelled physical violence, and many tweets are *about* GBV (jokes, news, opinions, TV plots) rather than disclosures.

**Implications for modelling.** Tasks that hinge on word order and the scope of a cue favour models that encode **context** — bidirectional RNNs and self-attention — over bag-of-words. Label noise puts a ceiling on achievable scores and means some "errors" will actually be annotation errors, which we will look for in Section 4.

### 1.7 Language variation and code-switching

The corpus is predominantly English, but reflects African and other global Englishes. We use a simple marker-word heuristic to estimate how often other languages or varieties appear (this is indicative, not a full language identification).

In [ ]:
MARKERS = {
    "Nigerian Pidgin": r"\b(abi|dey|wetin|abeg|wahala|oga|una|comot|sef|ehn)\b",
    "Swahili": r"\b(sana|kwa|hii|mimi|wewe|yeye|lakini|kama|hapa|pole)\b",
    "Tagalog/Filipino": r"\b(hindi|naman|ako|sya|siya|yung|mga|lang|talaga|kasi)\b",
    "Spanish": r"\b(que|pero|porque|muy|está|mujer)\b",
}
lang = pd.DataFrame({k: lower.str.contains(p) for k, p in MARKERS.items()})
summary = pd.DataFrame({"tweets": lang.sum(), "% of train": (lang.mean() * 100).round(2)})
display(summary)
print("Any non-English marker:", f"{lang.any(axis=1).mean():.2%}")
for k in MARKERS:
    ex = train[lang[k]].tweet.head(2).tolist()
    for e in ex:
        print(f"[{k}] {e[:180]}")

**What this reveals.** Only a small share of tweets (≈0.7% by this heuristic, mostly Nigerian Pidgin) contain code-switching into Pidgin, Tagalog, Spanish or other languages (e.g. *"Why would dey chill..\"he raped me\" na she go his house o"*). The heuristic also has false positives (e.g. the name *"hii"* matching Swahili), so the true figure is uncertain. These tweets are far too few to learn the languages from scratch, and the English-dominated word vocabulary will treat most of their tokens as rare or unknown.

**Implications for modelling.** Sub-word tokenisation and **multilingual / African-language pre-trained Transformers** (e.g. mBERT, XLM-R, AfroXLMR) are better placed to handle this variation than models trained only on this corpus. We expect code-switched tweets to be over-represented among errors for the from-scratch models.

### 1.8 Summary: from data findings to modelling decisions

| Finding | Evidence | Modelling decision |
|---|---|---|
| Extreme class imbalance (82% vs <0.5%) | §1.2 | Macro-F1 as primary metric; class-weighted loss; stratified splits |
| Short sequences (≤68 words, ≤128 sub-words) | §1.3 | `max_len` 64 words / 128 sub-words, no meaningful truncation |
| Headline vs. narrative style differs by class | §1.3 | Models can exploit style; check in error analysis |
| Long-tail vocabulary, ~50% hapax, social-media noise | §1.4 | Light normalisation only; pre-trained embeddings; sub-word Transformer |
| Labels largely determined by collection keywords | §1.5 | Strong bag-of-words baselines needed; evaluate beyond accuracy; keyword-masking probe |
| Overlapping cues and figurative use require context | §1.6 | Bidirectional sequence models and self-attention |
| Label noise | §1.6 | Treat some errors as annotation issues; avoid over-interpreting small differences |
| Code-switching / African Englishes | §1.7 | Multilingual or African-language pre-trained Transformer |

## 2. Related Work and Sequential Model Selection

*(To be completed in Step 2.)*

## 3. Experimental Investigation

*(To be completed in Step 3.)*

## 4. Error Analysis and Limitations

*(To be completed in Step 4.)*